#### Models:
- Shadows classifier: LightGBM
    ```python
    LGBM = {
        "model": LGBMClassifier,
        "n_estimators": [200, 200, 50],
        "min_child_samples": [7, 7, 10],
        "num_leaves": [256, 256, 2],
        "colsample_bytree": [0.7, 0.7, 0.1],
        "results": [],
        "trials": []
    }
    ```
    nearly best performance (95% to the xgb's 97%) + by far smallest model

- Shadows model: LightGBM
    ```python
    LGBM = {
        "model": LGBMRegressor,
        "n_estimators": [450, 450, 50],
        "min_child_samples": [15, 15, 10],
        "num_leaves": [256, 256, 2],
        "colsample_bytree": [0.7, 0.7, 0.1],
        "results": [],
        "trials": []
    }
    ```
    again, nearly best performance while the model is small


- Clear model: LightGBM
    ```python
    LGBM = {
        "model": LGBMRegressor,
        "n_estimators": [450, 450, 50],
        "min_child_samples": [15, 15, 10],
        "num_leaves": [256, 256, 2],
        "colsample_bytree": [0.7, 0.7, 0.1],
        "results": [],
        "trials": []
    }
    ```

In [1]:
import cuml

from cuml.ensemble import RandomForestRegressor

from sklearn import ensemble
from sklearn.ensemble import ExtraTreesRegressor, ExtraTreesClassifier

import sklearn

import matplotlib.pyplot as plt

import xgboost as xgb
from lightgbm import LGBMRegressor, LGBMClassifier
import lightgbm as lgb

import torch

import numpy as np
import cupy as cp

import pandas as pd
from pathlib import Path
import gc

from sklearn.metrics import r2_score

import time

In [2]:
# prepare data

datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/shadows-detection-dataset/trees-gt-shadows.csv")

#making sure
datasetPath.is_file()

True

In [3]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0


In [4]:
target_col = dataset["grviOut"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0.26, 0.26, 0.26, ..., 0.26, 0.26, 0.26],
      shape=(332640,), dtype=float32)

In [5]:
inputs = dataset.drop("grviOut", axis=1)
inputs = inputs.drop("sunRoll", axis=1)
inputs = inputs.drop("camRoll", axis=1)

inputs.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunPitch,sunYaw,camPitch,camYaw,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,-71.945084,19.489212,-19.35,-0.0,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,-71.945084,19.489212,-19.35,-0.0,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,-71.945084,19.489212,-19.35,-0.0,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,-71.945084,19.489212,-19.35,-0.0,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,-71.945084,19.489212,-19.35,-0.0,0.0


In [6]:
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 2.4851234e-01,  1.4691126e-04,  2.0387112e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       [ 2.0935635e-01,  2.5367978e-04,  3.4656093e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       [ 2.2862221e-01,  2.2864803e-04,  3.1710259e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       ...,
       [ 8.0871526e-03,  3.2670033e-01,  3.1690043e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00],
       [ 1.2527064e-02,  2.7300212e-01,  2.6233262e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00],
       [ 2.2494195e-02,  2.6528352e-01,  2.5932705e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00]],
      shape=(332640, 11), dtype=float32)

In [7]:
n_total = len(inputs)

n_train = int(0.9*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]

In [8]:
mask_high = val_inputs[:, 10] == 0.0
mask_low = ~mask_high

val_inputs_high = val_inputs[mask_high]
val_labels_high = val_labels[mask_high]

val_inputs_low = val_inputs[mask_low]
val_labels_low = val_labels[mask_low]

val_inputs = np.delete(val_inputs, 10, axis=1)
val_inputs_high = np.delete(val_inputs_high, 10, axis=1)
val_inputs_low = np.delete(val_inputs_low, 10, axis=1)

In [9]:
mask_high = train_inputs[:, 10] == 0.0
mask_low = ~mask_high

train_inputs_high = train_inputs[mask_high]
train_labels_high = train_labels[mask_high]

train_inputs_low = train_inputs[mask_low]
train_labels_low = train_labels[mask_low]

train_inputs = np.delete(train_inputs, 10, axis=1)
train_inputs_high = np.delete(train_inputs_high, 10, axis=1)
train_inputs_low = np.delete(train_inputs_low, 10, axis=1)

In [10]:
LGBMlow = LGBMRegressor(n_estimators = 450, min_child_samples = 15, num_leaves = 255, colsample_bytree = 0.7)
LGBMhigh = LGBMRegressor(n_estimators = 450, min_child_samples = 15, num_leaves = 255, colsample_bytree = 0.7)

In [11]:
# train shadows model

modelLGBMlow = LGBMlow.fit(train_inputs_low, train_labels_low)

modelLGBMlow.booster_.save_model("lgbm_model_low.txt")

y_pred = modelLGBMlow.predict(val_inputs_low)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred))
r2 = r2_score(val_labels_low, y_pred)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())

print ("Low RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2: " + str(r2))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002007 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1560
[LightGBM] [Info] Number of data points in the train set: 65866, number of used features: 10
[LightGBM] [Info] Start training from score 0.309761
Low RelRMSE: 3.5509729301295%, r2: 0.9679561254686334


In [12]:
# train clear model

modelLGBMhigh = LGBMhigh.fit(train_inputs_high, train_labels_high)

modelLGBMhigh.booster_.save_model("lgbm_model_high.txt")

y_pred = modelLGBMhigh.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred))
r2 = r2_score(val_labels_high, y_pred)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())

print ("High RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2: " + str(r2))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006014 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1631
[LightGBM] [Info] Number of data points in the train set: 233510, number of used features: 10
[LightGBM] [Info] Start training from score 0.304363
High RelRMSE: 5.823351046388488%, r2: 0.9252421531114401


In [13]:
shadowsClass = lgb.Booster(model_file="lgbm_model.txt")

In [14]:
results = []

for sample in val_inputs:
    if (round(shadowsClass.predict(np.array(sample).reshape(1, -1)).item())):
        results.append(modelLGBMlow.predict(np.array(sample).reshape(1, -1)).item())
    else:
        results.append(modelLGBMhigh.predict(np.array(sample).reshape(1, -1)).item())


rmse = np.sqrt(cuml.metrics.mean_squared_error(np.array(val_labels).reshape(-1), np.array(results, dtype=float).reshape(-1)))
r2 = r2_score(val_labels, np.array(results, dtype=float).reshape(-1))

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())

print ("RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2: " + str(r2))

RelRMSE: 5.771444728501314%, r2: 0.9244268312948692


In [15]:
# counting

import time

In [16]:
from cuml import ForestInference


inference_times_CPU = []
for i in range(0, 100):
    start_time = time.time()
    
    pred = round(shadowsClass.predict(np.array(sample).reshape(1, -1)).item())
    
    if (pred):
        results.append(modelLGBMlow.predict(np.array(sample).reshape(1, -1)).item())
    else:
        results.append(modelLGBMhigh.predict(np.array(sample).reshape(1, -1)).item())

    end_time = time.time()

    inference_times_CPU.append(end_time - start_time)

inference_times_CPU = np.array(inference_times_CPU, dtype = "float32")
print (f"CPU time: {inference_times_CPU.mean()} ({inference_times_CPU.std()})")


fil_model_LGBM_low = ForestInference.load(
    "lgbm_model_low.txt",
    model_type="lightgbm",
    is_classifier=False,
    output_type="cupy"
)

fil_model_LGBM_high = ForestInference.load(
    "lgbm_model_high.txt",
    model_type="lightgbm",
    is_classifier=False,
    output_type="cupy"
)

fil_shadowsClass = ForestInference.load(
    "lgbm_model.txt",
    model_type="lightgbm",
    is_classifier=True,
    output_type="cupy"
)

with cuml.using_output_type("cupy"):

    for i in range(0, 20):
        x_gpu = cp.asarray(val_inputs[i:i+1], dtype=cp.float32)
        pred = fil_model_LGBM_low.predict(x_gpu)
        pred_cpu = cp.asnumpy(pred)


    inference_times_GPU = []
    cputogpu = []
    gputocpu = []

    overall = []
    for i in range(0, 100):
        cp.cuda.Device().synchronize()
        general_start = time.time()
        start_time = time.time()

        x_gpu = cp.asarray(val_inputs[i:i+1], dtype=cp.float32)

        cp.cuda.Device().synchronize()
        end_time = time.time()
        
        t1 = end_time - start_time

        #########################################################

        cp.cuda.Device().synchronize()
        start_time = time.time()
        
        pred = cp.round(fil_shadowsClass.predict(x_gpu))
        
        if (pred):
            pred = fil_model_LGBM_low.predict(x_gpu)
        else:
            pred = fil_model_LGBM_high.predict(x_gpu)

        cp.cuda.Device().synchronize()
        end_time = time.time()

        t2 = end_time - start_time

        #########################################################

        cp.cuda.Device().synchronize()
        start_time = time.time()
        
        pred_cpu = cp.asnumpy(pred)

        cp.cuda.Device().synchronize()
        general_end = time.time()
        end_time = time.time()

        t3 = end_time - start_time

        #########################################################      

        cputogpu.append(t1)
        inference_times_GPU.append(t2)
        gputocpu.append(t3)

        overall.append(general_end - general_start)

    cputogpu = np.array(cputogpu, dtype = "float32")
    inference_times_GPU = np.array(inference_times_GPU, dtype = "float32")
    gputospu = np.array(gputocpu, dtype = "float32")

    overall = np.array(overall, dtype = "float32")
    
print(f"CPU to GPU time: {cputogpu.mean()} ({cputogpu.std()})")
print(f"GPU time: {inference_times_GPU.mean()} ({inference_times_GPU.std()})")
print(f"GPU to CPU time: {gputospu.mean()} ({gputospu.std()})")

print(f"Overall: {overall.mean()} ({overall.std()})")

CPU time: 0.0005032491753809154 (9.347106970380992e-05)
CPU to GPU time: 0.00017779112386051565 (0.00020069594029337168)
GPU time: 0.0013611364411190152 (0.00038817714084871113)
GPU to CPU time: 2.276897430419922e-05 (2.4389883037656546e-06)
Overall: 0.0015653776936233044 (0.0004520477377809584)
